# Kafka Demo

## Connect to the Kafka Broker Before Running the Notebook

Retrieve the connection details and credentials from the Canvas entry for this lab.
From the lab directory, run the connection helper with the non-secret values supplied in Canvas.

```
./connect-kafka <user>@<remote_server> <remote_port>
```

Enter the SSH and Kafka passwords only when prompted. Continue when the helper reports that it is connected.
It stores the Kafka settings privately outside this repository; do not put either password in the notebook.
Run `./disconnect-kafka` in a terminal when you finish the lab.

---

## Setup

The Codespace and DevContainer create `.venv` and install the notebook dependencies automatically.
Select `.venv/bin/python` as the notebook kernel if it is not already selected.
Use the following commands only when working outside the provided environment.
```
python -m venv <environment_name>
source <environment_name>/bin/activate  # On Windows: <environment_name>\Scripts\activate
```

Then install the requirements:
```
pip install -r requirements.txt
```
Or manually:
```
pip install kafka-python
```

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer
from typing import Dict, Any

# [TODO]: Fill in a unique identifier so your topic doesn't collide with others'
# Replace ... with your andrew_id as a string (e.g., "asmith") or any unique identifier
andrew_id = ...  # Example: andrew_id = "asmith"
topic = f"lab-kafka-{andrew_id}"
config_dir = Path(os.environ.get("XDG_CONFIG_HOME", Path.home() / ".config"))
config_path = config_dir / "mlip-kafka.conf"
if not config_path.exists():
    raise FileNotFoundError("Run ./connect-kafka in a terminal before starting the notebook.")

client_config = {}
for line in config_path.read_text().splitlines():
    if line and not line.startswith("#"):
        key, value = line.split("=", 1)
        client_config[key] = value

bootstrap_servers = [client_config["bootstrap.servers"]]
kafka_auth = {
    "security_protocol": client_config["security.protocol"],
    "sasl_mechanism": client_config["sasl.mechanism"],
    "sasl_plain_username": client_config["sasl.username"],
    "sasl_plain_password": client_config["sasl.password"],
}
print(f"Topic: {topic}")

### Producer Mode -> Writes Data to Broker

In [ ]:
# Below schema is for messages. You may change the city data if you wish but it is optional.
def make_city_data(city: str, temperature_f: str) -> Dict[str, Any]:
    return {
        "city": city,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature_f": temperature_f, # temperature in fahrenheit
    }

In [ ]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Kafka expects messages as bytes. Explore the documentation and decide how to serialize Python dict objects into bytes.
# Hint: You may want to convert your Python dict → JSON string → UTF-8 bytes.

producer = KafkaProducer(bootstrap_servers=bootstrap_servers,
                        value_serializer=...,
                        **kafka_auth)

In [ ]:
# [TODO]: Add a few more examples of city data below
cities = [make_city_data("Pittsburgh" , 64), make_city_data(... , ...), make_city_data(... , ...)]

print("Writing to Kafka Broker")
for i in range(10):
    data = cities[randint(0,len(cities)-1)] # random selection
    producer.send(topic=topic, value=data)
    sleep(1)

producer.flush()
print(f"Data written to topic: {topic}")

### Consumer Mode -> Reads Data from Broker

In [ ]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Choose an auto_offset_reset value: try 'earliest' to read from the beginning or 'latest' for new messages only.
# Note: Since producer uses value_serializer, message.value is bytes. We decode and parse JSON.

consumer = KafkaConsumer(
    topic,
    bootstrap_servers=bootstrap_servers,
    auto_offset_reset=...,  # [TODO]: Try 'earliest', 'latest', or 'none'
    group_id=topic,
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000,
    # Stop after 10 seconds without a new message so the cell completes on its own
    consumer_timeout_ms=10000,
    **kafka_auth
)

print('Reading Kafka Broker')
for message in consumer:
    # Producer serialized to JSON bytes, so we decode and parse
    message_str = message.value.decode('utf-8')
    message_dict = loads(message_str)
    print(message_dict)
    os.system(f"echo {message_str} >> kafka_log.csv")

# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
# Use kcat to consume the first 5 messages from your topic. The output includes each message's offset.
!kcat -F {config_path} -t {topic} -C -o earliest -c 5 -f "%o: %s\n"
# 
# Where:
# - -F: private client configuration, including the broker address
# - -t: your topic name (e.g., "lab-kafka-asmith")
# - -C: consumer mode
# - -o earliest: start from earliest offset
# - -c 5: consume 5 messages
# - -f "%o: %s\n": format to show offset and message